In [4]:
import os
import re
import ast
import torch
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, average_precision_score
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from tqdm import tqdm

# ==========================================
# 0. 环境初始化与挂载
# ==========================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("未检测到 Colab 环境，跳过云盘挂载。")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🚀 当前计算设备: {device}")

# ==========================================
# 1. 全局核心参数配置 (锁定 Top-10)
# ==========================================
TOP_K_LABELS = 10         # 🌟 强行锁定输出维度为 10
MODEL_CHECKPOINT = "facebook/esm2_t6_8M_UR50D"
BATCH_SIZE = 8
EPOCHS = 1               # 数据维度小了，5轮通常能看到明显收敛
MAX_SEQ_LENGTH = 1000

# 路径配置
PROJECT_DIR = "/content/drive/MyDrive/Protein_Project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# 请确保这里是你之前保存的那个包含 16931 条数据的 CSV 文件路径
RAW_DATA_PATH = os.path.join(PROJECT_DIR, "uniprot_proteins_30000_cleaned.csv") # 或者是你实际保存的名字
TOP10_DATA_PATH = os.path.join(PROJECT_DIR, f"uniprot_proteins_top{TOP_K_LABELS}.csv")

# ==========================================
# 2. 数据清洗模块：提取 Top-10 标签
# ==========================================
def process_top_k_data(raw_csv_path, top_k_csv_path, top_k):
    if os.path.exists(top_k_csv_path):
        print(f"\n📦 检测到已存在的 Top-{top_k} 数据集，直接加载...")
        df = pd.read_csv(top_k_csv_path)
        if isinstance(df['filtered_go_terms'].iloc[0], str):
            df['filtered_go_terms'] = df['filtered_go_terms'].apply(ast.literal_eval)
        return df

    print(f"\n⚙️ 尚未检测到 Top-{top_k} 数据集，开始从原始数据中提取...")
    df = pd.read_csv(raw_csv_path)
    
    # 解析可能被存为字符串的列表
    if isinstance(df['go_terms'].iloc[0], str):
        df['go_terms'] = df['go_terms'].apply(ast.literal_eval)
        
    all_go_terms = [term for terms in df['go_terms'] for term in terms]
    term_counts = Counter(all_go_terms)
    
    top_k_items = term_counts.most_common(top_k)
    valid_terms = {term for term, count in top_k_items}
    
    print(f"\n--- 📊 Top {top_k} 保留标签分布 ---")
    for term, count in top_k_items:
        print(f"标签 {term}: 出现 {count} 次")
    print("----------------------------------")
    
    # 过滤数据
    df['filtered_go_terms'] = df['go_terms'].apply(lambda terms: [t for t in terms if t in valid_terms])
    df = df[df['filtered_go_terms'].map(len) > 0].copy()
    
    df.to_csv(top_k_csv_path, index=False)
    print(f"✅ Top-{top_k} 数据集处理完成！有效样本数: {len(df)}")
    print(f"💾 已保存至: {top_k_csv_path}")
    return df

# ==========================================
# 3. Dataset 定义
# ==========================================
class ProteinFunctionDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_length):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            seq, padding='max_length', truncation=True, 
            max_length=self.max_length, return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.float32)
        return item

# ==========================================
# 4. 主流程开始：数据流向编排
# ==========================================
# 4.1 获取精准数据
df_top10 = process_top_k_data(RAW_DATA_PATH, TOP10_DATA_PATH, TOP_K_LABELS)

# 4.2 多热编码
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(df_top10['filtered_go_terms'])
num_labels = len(mlb.classes_)
print(f"\n确认模型输出维度 (num_labels): {num_labels}")

# 4.3 划分训练集和验证集 (80% 训练, 20% 评估)
train_seqs, val_seqs, train_labels, val_labels = train_test_split(
    df_top10['Sequence'].tolist(), 
    y_encoded, 
    test_size=0.2, 
    random_state=42 # 固定随机种子以保证结果公平
)

# 4.4 初始化 Tokenizer 与 DataLoader
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
train_dataset = ProteinFunctionDataset(train_seqs, train_labels, tokenizer, MAX_SEQ_LENGTH)
val_dataset = ProteinFunctionDataset(val_seqs, val_labels, tokenizer, MAX_SEQ_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

import torch.nn as nn

print("\n🧠 正在初始化 ESM-2 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=num_labels, problem_type="multi_label_classification"
)
model = model.to(device)

# ==========================================
# 💡 药方一：差分学习率 (Differential Learning Rates)
# ==========================================
# 让预训练底层保持极小的学习率 (1e-5)，防止学到的生物学知识被破坏
# 让顶层全新的分类头使用较大的学习率 (1e-3)，加速收敛
optimizer_grouped_parameters = [
    {"params": model.esm.parameters(), "lr": 1e-5},
    {"params": model.classifier.parameters(), "lr": 1e-3}
]
optimizer = AdamW(optimizer_grouped_parameters)

# ==========================================
# 💡 药方二：计算正样本权重 (Positive Weights)
# ==========================================
# 统计训练集中 0 和 1 的比例，给数量少的 1 (正样本) 赋予更高的权重
# 惩罚模型那种“无脑猜 0”的偷懒行为
total_samples = train_labels.shape[0]
pos_counts = train_labels.sum(axis=0)
neg_counts = total_samples - pos_counts
# 防止除以 0 的安全处理
pos_weights = torch.tensor(neg_counts / (pos_counts + 1e-5), dtype=torch.float32).to(device)

# 显式定义自带权重的损失函数，不再依赖 Hugging Face 的默认 Loss
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

print("\n🔥 开始带有动态权重和差分学习率的微调训练...")
for epoch in range(EPOCHS):
    model.train() 
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # 前向传播 (不直接使用 outputs.loss，而是手动计算)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        
        # 使用带有 pos_weight 的自定义损失函数
        loss = criterion(outputs.logits, labels)
        
        total_train_loss += loss.item()
        loss.backward()
        
        # 梯度裁剪 (Gradient Clipping)：防止梯度爆炸导致 Loss 剧烈震荡
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    print(f"--- Epoch {epoch+1} 结束 | 平均训练损失: {total_train_loss / len(train_loader):.4f} ---")

# 训练结束保存模型
final_save_path = os.path.join(PROJECT_DIR, f"esm2_top10_finetuned")
model.save_pretrained(final_save_path)
tokenizer.save_pretrained(final_save_path)
print(f"\n💾 最终模型已归档至云盘: {final_save_path}")

# ==========================================
# 6. 验证与科学指标计算 (直接连贯执行)
# ==========================================
print("\n🔍 正在验证集上评估模型真实表现...")
model.eval()
all_preds = []
all_labels_list = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="[Eval]"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = torch.sigmoid(logits).cpu().numpy()
        
        all_preds.append(probs)
        all_labels_list.append(labels.cpu().numpy())

all_preds = np.vstack(all_preds)
all_labels_matrix = np.vstack(all_labels_list)

# 动态寻找最佳阈值而不是写死 0.5 (针对不平衡数据极其有效)
best_threshold = 0.5
best_macro_f1 = 0
# 在 0.1 到 0.9 之间扫描最佳阈值
for th in np.arange(0.1, 0.9, 0.1):
    binary_preds_tmp = (all_preds > th).astype(int)
    f1_tmp = f1_score(all_labels_matrix, binary_preds_tmp, average='macro', zero_division=0)
    if f1_tmp > best_macro_f1:
        best_macro_f1 = f1_tmp
        best_threshold = th

binary_preds = (all_preds > best_threshold).astype(int)
macro_f1 = f1_score(all_labels_matrix, binary_preds, average='macro', zero_division=0)
micro_f1 = f1_score(all_labels_matrix, binary_preds, average='micro', zero_division=0)
macro_auprc = average_precision_score(all_labels_matrix, all_preds, average='macro')

print("\n" + "=".center(50, "="))
print("🏆 验证集终极评估报告 (Top-10 标签)".center(45))
print("=".center(50, "="))
print(f"最佳分类阈值 (Best Threshold): {best_threshold:.2f}")
print(f"Macro F1-Score: {macro_f1:.4f}")
print(f"Micro F1-Score: {micro_f1:.4f}")
print(f"Macro AUPRC:    {macro_auprc:.4f}")
print("=".center(50, "="))
print("如果 Macro F1 显著提升，说明模型已成功掌握核心功能映射！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

🚀 当前计算设备: cuda

📦 检测到已存在的 Top-10 数据集，直接加载...

确认模型输出维度 (num_labels): 10

🧠 正在初始化 ESM-2 模型...


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmForSequenceClassification LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🔥 开始带有动态权重和差分学习率的微调训练...


Epoch 1/1 [Train]:  14%|█▍        | 201/1461 [03:19<20:51,  1.01it/s, loss=1.0699]


KeyboardInterrupt: 